# Deep Learning Project: Predicting if a bank customer will exit the bank

## Importing the libraries

In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

## Importing the dataset

In [4]:
dataset = pd.read_csv("Bank_Customers.csv")
X = dataset.iloc[:, 3:-1].values
y = dataset.iloc[:, -1].values

#print(X)
#print(Y)

## Data preprocessing

### Splitting the data into train & test

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### Replace missing data

In [7]:
from sklearn.impute import SimpleImputer

numerical_columns = [0, 3, 4, 5, 6, 9]
categorical_columns = [1, 2, 7, 8]

imputer = SimpleImputer(missing_values=np.nan, strategy="median")
imputer.fit(X_train[:, numerical_columns])
X_train[:, numerical_columns] = imputer.transform(X_train[:, numerical_columns])
X_test[:, numerical_columns] = imputer.transform(X_test[:, numerical_columns])

imputer = SimpleImputer(missing_values=np.nan, strategy="most_frequent")
imputer.fit(X_train[:, categorical_columns])
X_train[:, categorical_columns] = imputer.transform(X_train[:, categorical_columns])
X_test[:, categorical_columns] = imputer.transform(X_test[:, categorical_columns])


### One-hot encoding the geography & gender column

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

ct = ColumnTransformer(
    transformers=[("encoder", OneHotEncoder(sparse_output=False), [1, 2])],
    remainder="passthrough",
)
X_train = np.array(ct.fit_transform(X_train))
X_test = np.array(ct.transform(X_test))

### Feature scaling

In [9]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## Building the Nerual Network

#### A feedforward neural network is constructed with:


*   An input layer that receives the preprocessed feature vectors.
*   Two fully connected hidden layers (64 and 32 neurons) using the ReLU activation function.


*   One output layer with a single neuron using the Sigmoid activation function for binary classification.

#### The ReLU activation function is used in the hidden layers to introduce non-linearity and enable the network to learn complex patterns from the data. The Sigmoid activation function is applied in the output layer to produce probability values between 0 and 1, making it suitable for predicting binary outcomes.






### Initializing the ANN

In [11]:
model = tf.keras.models.Sequential()

### Adding the input layer and the first hidden layer

In [12]:
tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],))

<Dense name=dense_1, built=False>

### Adding the second hidden layer

In [13]:
model.add(tf.keras.layers.Dense(units=32, activation='relu'))

### Adding the output layer

In [14]:
model.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

### Compiling the ANN

#### The model is compiled using:


*   Adam optimizer with learning rate 0.001 for efficient gradient updates.
*   Binary Crossentropy loss, suitable for binary classification tasks.
*   Evaluation metrics including Accuracy, Precision, Recall, and AUC to comprehensively assess performance.



In [15]:
tf.keras.backend.clear_session()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)